# 🫀 Heart Failure Risk Score Prediction
## Aditya's Role: Model Evaluation

**Models Evaluated:** RF, XGBoost, CatBoost, HRS  
**Outputs:**
- Table 11 — Model Performance Comparison
- Plot 12 — ROC Curves (all models)

---
### 📂 Upload these files to Colab before running:
From the GitHub repo:
- `Outputs/cleaned_test.csv`
- `Outputs/rf_model.pkl`
- `Outputs/xgb_model.pkl`
- `Outputs/catboost_model.pkl`
- `Tables/hrs_risk_categories.csv`

In [ ]:
# Cell 1 — Install libraries
import subprocess, sys

for pkg in ["xgboost", "catboost", "scikit-learn", "matplotlib", "seaborn", "pandas", "numpy", "joblib"]:
    subprocess.check_call([sys.executable, "-m", "pip", "install", pkg, "-q"])

print("✅ Libraries installed.")

In [ ]:
# Cell 2 — Imports
import os, warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from catboost import CatBoostClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, roc_curve, auc,
    classification_report
)

plt.rcParams.update({"figure.dpi": 150, "axes.spines.top": False, "axes.spines.right": False})
PALETTE = ["#D7191C", "#1A9641", "#FDAE61", "#7B2D8B"]

print("✅ All imports done.")

In [ ]:
# Cell 3 — Find uploaded files automatically
import glob

def find_file(filename):
    """Search entire filesystem for a filename and return its path."""
    for root, dirs, files in os.walk("/"):
        # Skip system dirs to speed up search
        dirs[:] = [d for d in dirs if d not in ['proc', 'sys', 'dev', 'run']]
        if filename in files:
            return os.path.join(root, filename)
    return None

print("🔍 Searching for files...")
DATA_PATH  = find_file("cleaned_test.csv")
HRS_PATH   = find_file("hrs_risk_categories.csv")
RF_PATH    = find_file("rf_model.pkl")
XGB_PATH   = find_file("xgb_model.pkl")
CAT_PATH   = find_file("catboost_model.pkl")

OUTPUT_DIR = "/content/Outputs/"
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f"cleaned_test.csv       : {DATA_PATH}")
print(f"hrs_risk_categories.csv: {HRS_PATH}")
print(f"rf_model.pkl           : {RF_PATH}")
print(f"xgb_model.pkl          : {XGB_PATH}")
print(f"catboost_model.pkl     : {CAT_PATH}")

In [ ]:
# Cell 4 — Load test data
df_test = pd.read_csv(DATA_PATH)
print(f"Test set shape: {df_test.shape}")
display(df_test.head(3))

# Auto-detect target column
TARGET_COL = "target"
if TARGET_COL not in df_test.columns:
    binary_cols = [c for c in df_test.columns
                   if df_test[c].nunique() == 2 and df_test[c].dtype in ["int64", "float64"]]
    TARGET_COL = binary_cols[-1]
    print(f"⚠️  Auto-detected target column: '{TARGET_COL}'")

X_test = df_test.drop(columns=[TARGET_COL])
y_test = df_test[TARGET_COL].astype(int)

print(f"\nFeatures : {X_test.shape[1]}")
print(f"Samples  : {len(y_test)}")
print(f"Positives: {y_test.sum()} ({y_test.mean()*100:.1f}%)")

In [ ]:
# Cell 5 — Load models
rf_model  = joblib.load(RF_PATH)
xgb_model = joblib.load(XGB_PATH)
cat_model = joblib.load(CAT_PATH)

print("✅ RF model loaded")
print("✅ XGBoost model loaded")
print("✅ CatBoost model loaded")

In [ ]:
# Cell 6 — Load HRS scores from Isha
hrs_df = pd.read_csv(HRS_PATH)
print(f"HRS file loaded — shape: {hrs_df.shape}")
print(hrs_df.head(3))

# Normalize HRS score to [0, 1]
hrs_scores = hrs_df["HRS_score"].values[:len(y_test)]
prob_hrs   = (hrs_scores - hrs_scores.min()) / (hrs_scores.max() - hrs_scores.min() + 1e-9)

# Binary prediction
if "risk_class" in hrs_df.columns:
    risk_map = {"Low": 0, "Moderate": 0, "High": 1}
    pred_hrs = hrs_df["risk_class"].map(risk_map).values[:len(y_test)]
else:
    pred_hrs = (prob_hrs >= 0.5).astype(int)

print("\n✅ HRS scores loaded.")

In [ ]:
# Cell 7 — Get predictions
prob_rf  = rf_model.predict_proba(X_test)[:, 1]
prob_xgb = xgb_model.predict_proba(X_test)[:, 1]
prob_cat = cat_model.predict_proba(X_test)[:, 1]

pred_rf  = rf_model.predict(X_test)
pred_xgb = xgb_model.predict(X_test)
pred_cat = cat_model.predict(X_test)

MODEL_NAMES = ["RF",     "XGBoost",  "CatBoost",  "HRS"    ]
PROBS       = [prob_rf,  prob_xgb,   prob_cat,    prob_hrs ]
PREDS       = [pred_rf,  pred_xgb,   pred_cat,    pred_hrs ]

print("✅ Predictions ready for all models.")

In [ ]:
# Cell 8 — Table 11: Model Performance Comparison
def evaluate(name, y_true, y_pred, y_prob):
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel()
    return {
        "Model"      : name,
        "Accuracy"   : round(accuracy_score(y_true, y_pred),                    4),
        "Precision"  : round(precision_score(y_true, y_pred, zero_division=0),  4),
        "Recall"     : round(recall_score(y_true, y_pred, zero_division=0),     4),
        "F1 Score"   : round(f1_score(y_true, y_pred, zero_division=0),         4),
        "AUC-ROC"    : round(roc_auc_score(y_true, y_prob),                     4),
        "Sensitivity": round(tp / max(tp + fn, 1), 4),
        "Specificity": round(tn / max(tn + fp, 1), 4),
        "TP": int(tp), "FP": int(fp), "TN": int(tn), "FN": int(fn)
    }

rows = [evaluate(n, y_test.values, p, pr) for n, p, pr in zip(MODEL_NAMES, PREDS, PROBS)]
table11 = pd.DataFrame(rows).set_index("Model")

# Save CSV
table11.to_csv(os.path.join(OUTPUT_DIR, "Table11_Model_Performance_Comparison.csv"))

print("📋 Table 11: Model Performance Comparison")
display(
    table11.style
    .background_gradient(subset=["AUC-ROC", "F1 Score", "Recall"], cmap="YlGn")
    .background_gradient(subset=["Accuracy", "Precision"], cmap="Blues")
    .format("{:.4f}", subset=["Accuracy","Precision","Recall","F1 Score","AUC-ROC","Sensitivity","Specificity"])
)
print("\n✅ Table 11 saved.")

In [ ]:
# Cell 9 — Confusion Matrices
fig, axes = plt.subplots(1, 4, figsize=(18, 4))

for ax, name, pred, cmap in zip(axes, MODEL_NAMES, PREDS, ["Greens", "Oranges", "Purples", "Reds"]):
    cm = confusion_matrix(y_test, pred, labels=[0, 1])
    sns.heatmap(cm, annot=True, fmt="d", cmap=cmap, ax=ax,
                xticklabels=["Pred 0", "Pred 1"],
                yticklabels=["Actual 0", "Actual 1"],
                linewidths=0.5, linecolor="white",
                cbar=False, annot_kws={"size": 13, "weight": "bold"})
    tn, fp, fn, tp = cm.ravel()
    acc = accuracy_score(y_test, pred)
    ax.set_title(f"{name}\nAcc={acc:.3f}  TP={tp}  FN={fn}", fontsize=10, pad=8)
    ax.set_xlabel("Predicted", fontsize=9)
    ax.set_ylabel("Actual", fontsize=9)

fig.suptitle("Confusion Matrices — All Models", fontsize=14, fontweight="bold", y=1.03)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "Confusion_Matrices_All_Models.png"), dpi=150, bbox_inches="tight")
plt.show()
print("✅ Confusion matrices saved.")

In [ ]:
# Cell 10 — Plot 12: ROC Curves
fig, ax = plt.subplots(figsize=(8, 7))

ax.plot([0, 1], [0, 1], "k--", lw=1.2, label="Random (AUC = 0.50)")

for name, prob, color in zip(MODEL_NAMES, PROBS, PALETTE):
    fpr, tpr, _ = roc_curve(y_test, prob)
    roc_auc     = auc(fpr, tpr)
    ax.plot(fpr, tpr, lw=2.5, color=color, label=f"{name}  (AUC = {roc_auc:.4f})")

ax.fill_between([0, 1], [0, 1], alpha=0.05, color="gray")
ax.set_xlabel("False Positive Rate (1 - Specificity)", fontsize=12)
ax.set_ylabel("True Positive Rate (Sensitivity)", fontsize=12)
ax.set_title("Plot 12: ROC Curves — All Models", fontsize=13, pad=12)
ax.legend(loc="lower right", fontsize=10, framealpha=0.9)
ax.set_xlim(0, 1)
ax.set_ylim(0, 1.02)

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "Plot12_ROC_Curves.png"), dpi=150, bbox_inches="tight")
plt.show()
print("✅ Plot 12 saved.")

In [ ]:
# Cell 11 — Classification Reports
for name, pred in zip(MODEL_NAMES, PREDS):
    print(f"{'='*50}")
    print(f"  {name} — Classification Report")
    print(f"{'='*50}")
    print(classification_report(y_test, pred, target_names=["No Disease", "Disease"]))
    print()

In [ ]:
# Cell 12 — Summary
print("=" * 60)
print("  📁 ALL OUTPUTS SAVED TO:", os.path.abspath(OUTPUT_DIR))
print("=" * 60)
for f in sorted(os.listdir(OUTPUT_DIR)):
    size = os.path.getsize(os.path.join(OUTPUT_DIR, f))
    print(f"  ✅  {f:55s}  ({size:,} bytes)")

best = table11["AUC-ROC"].idxmax()
print(f"\n🏆 Best model by AUC-ROC: {best} ({table11.loc[best,'AUC-ROC']:.4f})")